# POC: shared DINOv3 + DPT for segmentation and depth

This notebook is deliberately comparable with `dinov3_dpt_seg_depth.ipynb`: it uses the same LuSNAR loader, scene split, image size, batch size, DINOv3 checkpoint discovery, DPT-style feature fusion, losses, and metrics.

The difference is that one DINOv3 backbone and one DPT neck are shared by two task heads, trained jointly.

In [1]:

import socket
import torch

print("host:", socket.gethostname())
print("torch:", torch.__file__)
print("cuda available:", torch.cuda.is_available())

host: p206.uppmax.uu.se
torch: /gorilla/proj/uppmax2026-1-299/taiko_sandbox_please_work/.venv_lusnar/lib/python3.12/site-packages/torch/__init__.py
cuda available: True


In [2]:
from pathlib import Path
import contextlib, gc, json, math, re, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm


In [3]:
ROOT = Path("/proj/uppmax2026-1-299/task1_groupc_dataset/lusnar/extracted")
UPPMAX_CODE_DIR = Path("/proj/uppmax2026-1-299/task1_groupc_dataset")
def _find_code_dir():
    candidates = []
    if "__vsc_ipynb_file__" in globals(): candidates.append(Path(__vsc_ipynb_file__).parent)
    candidates += [Path.cwd(), *Path.cwd().parents]
    for p in candidates:
        if (p / "lusnar_dataset.py").exists(): return p
    raise FileNotFoundError("Could not find lusnar_dataset.py")
CODE_DIR = UPPMAX_CODE_DIR if (UPPMAX_CODE_DIR / "lusnar_dataset.py").exists() else _find_code_dir()
import sys
sys.path.insert(0, str(CODE_DIR))
import lusnar_dataset as L

IMAGE_SIZE, BATCH_SIZE, NUM_WORKERS = 512, 8, 4
MAX_DEPTH = None
VAL_SCENES, TEST_SCENES, N_VAL, N_TEST = ["Moon_8"], ["Moon_9", "Moon_3"], 1, 1
INDEX_CACHE = Path.home() / "lusnar_cache" / "index.csv"
SEED = 0
OUT_LAYERS = (2, 5, 8, 11)
DPT_FEATURES, N_DEPTH_BINS = 256, 64
USE_CLS_READOUT = True
UNFREEZE_LAST_N = 0
HEAD_LR, BACKBONE_LR, WEIGHT_DECAY = 2e-4, 1e-5, 1e-2
EPOCHS, SEG_LOSS_WEIGHT, DEPTH_LOSS_WEIGHT = 10, 1.0, 1.0
RUN_DIR = Path.home() / "lusnar_runs" / "dpt_shared_multitask_poc"
DINO_ID = "facebook/dinov3-vitb16-pretrain-lvd1689m"
DINO_WEIGHTS_NAME = "dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"
DINO_WEIGHTS = next((p for p in [UPPMAX_CODE_DIR / DINO_WEIGHTS_NAME, CODE_DIR / DINO_WEIGHTS_NAME, Path.home() / "Downloads" / DINO_WEIGHTS_NAME] if p.exists()), None)
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.bfloat16 if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16
def autocast(): return torch.autocast("cuda", dtype=AMP_DTYPE) if DEVICE.type == "cuda" else contextlib.nullcontext()
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("CODE_DIR:", CODE_DIR, "\nROOT:", ROOT, "\nDINO_WEIGHTS:", DINO_WEIGHTS, "\ndevice:", DEVICE)

CODE_DIR: /proj/uppmax2026-1-299/task1_groupc_dataset 
ROOT: /proj/uppmax2026-1-299/task1_groupc_dataset/lusnar/extracted 
DINO_WEIGHTS: /proj/uppmax2026-1-299/task1_groupc_dataset/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth 
device: cuda


In [4]:
# Same dataset setup as the existing notebook.
if not ROOT.exists() and (CODE_DIR / "Moon_1").exists():
    ROOT = CODE_DIR / "Moon_1"
    VAL_SCENES, TEST_SCENES, N_VAL, N_TEST = [], [], 0, 0
    INDEX_CACHE = None
cfg = L.LoaderConfig(root=str(ROOT), image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
                     num_workers=NUM_WORKERS, max_depth=MAX_DEPTH,
                     val_scenes=VAL_SCENES, test_scenes=TEST_SCENES, n_val=N_VAL, n_test=N_TEST,
                     augment_train=True, index_cache=str(INDEX_CACHE) if INDEX_CACHE else None, seed=SEED)
loaders, datasets = L.make_dataloaders(cfg)

# Match the existing notebook's fallback when only one local scene is available.
if "val" not in datasets:
    frac = 0.15
    idx = datasets["train"].idx
    pos = idx.groupby("scene").cumcount()
    cut = (idx.groupby("scene")["stem"].transform("size") * (1 - frac)).astype(int)
    tr_idx, va_idx = idx[pos < cut], idx[pos >= cut]
    datasets["train"] = L.LuSNARDataset(tr_idx, image_size=IMAGE_SIZE, max_depth=MAX_DEPTH, augment=True)
    datasets["val"] = L.LuSNARDataset(va_idx, image_size=IMAGE_SIZE, max_depth=MAX_DEPTH, augment=False)
    def make_loader(ds, train):
        return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=train, drop_last=train, num_workers=NUM_WORKERS,
                          pin_memory=torch.cuda.is_available(), persistent_workers=NUM_WORKERS > 0,
                          worker_init_fn=L._seed_worker, generator=torch.Generator().manual_seed(SEED))
    loaders["train"], loaders["val"] = make_loader(datasets["train"], True), make_loader(datasets["val"], False)
train_loader, val_loader, test_loader = loaders["train"], loaders["val"], loaders.get("test")
print({k: len(v.idx) for k, v in datasets.items()})

counts = L.class_pixel_counts(datasets["train"], n_samples=200)
seg_class_weights = L.class_weights(counts)
vals = []
for i in np.random.default_rng(SEED).choice(len(datasets["train"]), size=min(50, len(datasets["train"])), replace=False):
    d = L.load_depth(datasets["train"].idx.iloc[int(i)]["depth"])
    d = d[np.isfinite(d) & (d > 0) & (d < L.DEPTH_SKY_VALUE)]
    if MAX_DEPTH is not None: d = d[d <= MAX_DEPTH]
    vals.append(d[::max(1, d.size // 20000)])
vals = np.concatenate(vals)
pct = {p: float(np.percentile(vals, p)) for p in (0.1, 99.9)}
DEPTH_MIN = max(0.05, 0.8 * pct[0.1])
DEPTH_MAX = MAX_DEPTH if MAX_DEPTH is not None else 1.2 * pct[99.9]
print("depth bins:", DEPTH_MIN, DEPTH_MAX, "classes:", L.CLASS_NAMES)

train:  8,338 frames | scenes ['Moon_1', 'Moon_2', 'Moon_4', 'Moon_5', 'Moon_6', 'Moon_7']
val  :  1,351 frames | scenes ['Moon_8']
test :  3,317 frames | scenes ['Moon_3', 'Moon_9']
{'train': 8338, 'val': 1351, 'test': 3317}


/gorilla/proj/uppmax2026-1-299/taiko_sandbox_please_work/.venv_lusnar/lib/python3.12/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


depth bins: 1.0523437500000001 1681.2 classes: ['Lunar Regolith', 'Impact Crater', 'Rock', 'Mountain', 'Sky']


## Shared model

The backbone and DPT neck are instantiated once. The segmentation and depth heads are separate.

In [5]:
from transformers import AutoModel, DINOv3ViTConfig, DINOv3ViTModel
VITB16_CFG = dict(hidden_size=768, num_hidden_layers=12, num_attention_heads=12, intermediate_size=3072, num_register_tokens=4, patch_size=16)

def convert_meta_dinov3(sd):
    out = {}
    for k, v in sd.items():
        v = v.float()
        if k == "rope_embed.periods" or k.endswith("bias_mask"): continue
        m = re.match(r"blocks\.(\d+)\.(.*)", k)
        if m is None:
            k2 = {"cls_token": "embeddings.cls_token", "storage_tokens": "embeddings.register_tokens", "mask_token": "embeddings.mask_token", "patch_embed.proj.weight": "embeddings.patch_embeddings.weight", "patch_embed.proj.bias": "embeddings.patch_embeddings.bias"}.get(k, k)
            out[k2] = v.reshape(1, 1, -1) if k == "mask_token" else v
            continue
        i, rest = m.groups(); p = f"model.layer.{i}."
        if rest.startswith("attn.qkv."):
            kind = rest.split(".")[-1]
            if kind == "bias": v = v * sd[k + "_mask"].float()
            q, kk, vv = v.chunk(3, dim=0)
            out[p + f"attention.q_proj.{kind}"], out[p + f"attention.v_proj.{kind}"] = q, vv
            if kind == "weight": out[p + "attention.k_proj.weight"] = kk
            continue
        for a, b in [("attn.proj", "attention.o_proj"), ("mlp.fc1", "mlp.up_proj"), ("mlp.fc2", "mlp.down_proj"), ("ls1.gamma", "layer_scale1.lambda1"), ("ls2.gamma", "layer_scale2.lambda1")]: rest = rest.replace(a, b)
        out[p + rest] = v
    return out

def load_dinov3():
    if DINO_WEIGHTS is not None:
        m = DINOv3ViTModel(DINOv3ViTConfig(**VITB16_CFG))
        m.load_state_dict(convert_meta_dinov3(torch.load(DINO_WEIGHTS, map_location="cpu", weights_only=True)), strict=True)
        print("Loaded local pretrained weights")
        return m, True
    try:
        m = AutoModel.from_pretrained(DINO_ID)
        print("Loaded Hugging Face pretrained weights")
        return m, True
    except Exception as e:
        print("WARNING: random DINOv3 weights:", str(e).splitlines()[0])
        return DINOv3ViTModel(DINOv3ViTConfig(**VITB16_CFG)), False

def find_blocks(model):
    n = model.config.num_hidden_layers
    for _, mod in model.named_modules():
        if isinstance(mod, nn.ModuleList) and len(mod) == n: return list(mod)
    raise RuntimeError("DINO transformer blocks not found")


In [6]:

class DinoFeatures(nn.Module):
    def __init__(self, model):
        super().__init__(); self.model = model
        self.patch, self.embed_dim = model.config.patch_size, model.config.hidden_size
        self.n_prefix, self.blocks = 1 + model.config.num_register_tokens, find_blocks(model)
        self.cache = {}
        for i in OUT_LAYERS: self.blocks[i].register_forward_hook(self.hook(i))
    def hook(self, i):
        def f(_, __, output): self.cache[i] = output[0] if isinstance(output, (tuple, list)) else output
        return f
    def forward(self, x):
        B, _, H, W = x.shape; h, w = H // self.patch, W // self.patch
        frozen = not any(p.requires_grad for p in self.model.parameters())
        with torch.no_grad() if frozen else contextlib.nullcontext():
            self.cache = {}; self.model(pixel_values=x); outs = []
            for i in OUT_LAYERS:
                t = self.model.norm(self.cache[i]); patches, cls = t[:, self.n_prefix:], t[:, 0]
                outs.append((patches.transpose(1, 2).reshape(B, -1, h, w), cls))
        self.cache = {}; return outs

class RCU(nn.Module):
    def __init__(self, c): super().__init__(); self.a=nn.Conv2d(c,c,3,padding=1); self.b=nn.Conv2d(c,c,3,padding=1)
    def forward(self,x): return x + self.b(F.relu(self.a(F.relu(x))))

class Fuse(nn.Module):
    def __init__(self,c): super().__init__(); self.r1,self.r2=RCU(c),RCU(c); self.out=nn.Conv2d(c,c,1)
    def forward(self,x,skip=None,size=None):
        if skip is not None: x=x+self.r1(skip)
        x=self.r2(x); x=F.interpolate(x,size=size,mode="bilinear",align_corners=True) if size is not None else F.interpolate(x,scale_factor=2,mode="bilinear",align_corners=True)
        return self.out(x)

class DPTNeck(nn.Module):
    def __init__(self, embed=768, features=256):
        super().__init__()
        cs=(96,192,384,768)
        self.read=nn.ModuleList([nn.Sequential(nn.Linear(2*embed,embed),nn.GELU()) for _ in cs]) if USE_CLS_READOUT else None
        self.proj=nn.ModuleList([nn.Conv2d(embed,c,1) for c in cs])
        self.res=nn.ModuleList([nn.ConvTranspose2d(cs[0],cs[0],4,stride=4),
                                nn.ConvTranspose2d(cs[1],cs[1],2,stride=2),
                                nn.Identity(),
                                nn.Conv2d(cs[3],cs[3],3,stride=2,padding=1)])
        self.to_feat=nn.ModuleList([nn.Conv2d(c,features,3,padding=1,bias=False) for c in cs])
        self.fuse=nn.ModuleList([Fuse(features) for _ in cs])
    
    def forward(self, feats):
        layers=[]
        for i,(f,cls) in enumerate(feats):
            if self.read is not None:
                B,C,h,w=f.shape
                tok=f.flatten(2).transpose(1,2) 
                f=self.read[i](torch.cat([tok,cls[:,None].expand_as(tok)],-1)).transpose(1,2).reshape(B,C,h,w)
            layers.append(self.to_feat[i](self.res[i](self.proj[i](f))))

        l1,l2,l3,l4=layers

        p=self.fuse[3](l4,size=l3.shape[-2:])
        p=self.fuse[2](p,l3,size=l2.shape[-2:])
        p=self.fuse[1](p,l2,size=l1.shape[-2:])
        return self.fuse[0](p,l1)

class SegHead(nn.Module):
    def __init__(self): super().__init__(); self.net=nn.Sequential(nn.Conv2d(256,256,3,padding=1,bias=False),nn.BatchNorm2d(256),nn.ReLU(),nn.Dropout2d(.1),nn.Conv2d(256,L.NUM_CLASSES,1))
    def forward(self,x,size): return F.interpolate(self.net(x),size=size,mode="bilinear",align_corners=False)


class DepthHead(nn.Module):
    def __init__(self,dmin,dmax):
        super().__init__(); self.net=nn.Sequential(nn.Conv2d(256,128,3,padding=1),nn.ReLU(),nn.Conv2d(128,32,3,padding=1),nn.ReLU(),nn.Conv2d(32,N_DEPTH_BINS,1)); self.register_buffer("log_centers",torch.linspace(math.log(dmin),math.log(dmax),N_DEPTH_BINS).view(1,-1,1,1))
    def forward(self,x,size):
        p=self.net(x).float().softmax(1); return F.interpolate((p*self.log_centers).sum(1,keepdim=True).exp(),size=size,mode="bilinear",align_corners=False)

class SharedDPT(nn.Module):
    def __init__(self, dmin, dmax):
        super().__init__(); hf, self.pretrained=load_dinov3(); self.backbone=DinoFeatures(hf); self.neck=DPTNeck(self.backbone.embed_dim); self.seg=SegHead(); self.depth=DepthHead(dmin,dmax)
        self.backbone.model.requires_grad_(False)
        if UNFREEZE_LAST_N>0:
            for b in self.backbone.blocks[-UNFREEZE_LAST_N:]: b.requires_grad_(True)
            self.backbone.model.norm.requires_grad_(True)
    def forward(self,x):
        f=self.neck(self.backbone(x)); size=x.shape[-2:]; return {"seg":self.seg(f,size),"depth":self.depth(f,size)}


In [7]:
model=SharedDPT(DEPTH_MIN,DEPTH_MAX).to(DEVICE)
print("trainable parameters (M):", sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6)

Loaded local pretrained weights


trainable parameters (M): 25.381285


In [8]:
def grad_loss(pred, gt, valid):
    v=valid.bool() & (gt>1e-6)
    d=torch.where(v,torch.log(pred.clamp_min(1e-6))-torch.log(gt.clamp_min(1e-6)),torch.zeros_like(pred))
    mx,my=v[..., :,1:]&v[..., :, :-1],v[...,1:,:]&v[...,:-1,:]
    return ((d[..., :,1:]-d[...,:,:-1]).abs()[mx].sum()+(d[...,1:,:]-d[...,:-1,:]).abs()[my].sum())/(mx.sum()+my.sum()).clamp_min(1)

ce=nn.CrossEntropyLoss(weight=seg_class_weights.to(DEVICE),ignore_index=L.IGNORE_INDEX)

def losses(out,b):
    seg=ce(out["seg"],b["label"])
    valid=b["label"]!=L.IGNORE_INDEX
    prob=out["seg"].softmax(1)*valid[:,None]
    oh=F.one_hot(b["label"].clamp(max=L.NUM_CLASSES-1)*valid,L.NUM_CLASSES).permute(0,3,1,2).float()*valid[:,None]
    dice=1-((2*(prob*oh).sum((0,2,3))+1)/((prob+oh).sum((0,2,3))+1)).mean()
    v=b["depth_valid"].bool() & (b["depth"]>1e-6); g=torch.log(out["depth"].clamp_min(1e-6)[v])-torch.log(b["depth"][v])
    silog=torch.sqrt((g*g).mean()-.85*g.mean()**2+1e-6) if v.any() else out["depth"].sum()*0
    gr=grad_loss(out["depth"],b["depth"],b["depth_valid"])
    seg_loss=seg+0.5*dice
    depth_loss=silog+0.5*gr

    return SEG_LOSS_WEIGHT*seg_loss+DEPTH_LOSS_WEIGHT*depth_loss,{"seg":seg_loss.item(),"depth":depth_loss.item()}

def optimizer_for(m):
    groups={"head":[],"backbone":[]}
    for n,p in m.named_parameters():
        if p.requires_grad: groups["backbone" if n.startswith("backbone.") else "head"].append(p)
    return torch.optim.AdamW([
        {"params":groups["head"],"lr":HEAD_LR,"weight_decay":WEIGHT_DECAY},
        {"params":groups["backbone"],"lr":BACKBONE_LR,"weight_decay":WEIGHT_DECAY}], betas=(.9,.999))

@torch.no_grad()
def evaluate(m,loader,max_batches=None):
    m.eval()
    conf=torch.zeros(L.NUM_CLASSES,L.NUM_CLASSES,dtype=torch.long,device=DEVICE)
    absrel=[]
    rmse=[]
    for i,b in enumerate(loader):
        if max_batches is not None and i>=max_batches: break
        b={k:v.to(DEVICE) if torch.is_tensor(v) else v for k,v in b.items()}
        o=m(b["image"]); pred=o["seg"].argmax(1)
        mask=b["label"]!=L.IGNORE_INDEX
        conf += torch.bincount(b["label"][mask]*L.NUM_CLASSES+pred[mask],minlength=L.NUM_CLASSES**2).view(L.NUM_CLASSES,L.NUM_CLASSES)
        for j in range(pred.shape[0]):
            v=b["depth_valid"][j,0]
            if v.sum()>10:
                p,g=o["depth"][j,0][v].clamp_min(1e-3),b["depth"][j,0][v]
                absrel.append(float(((p-g).abs()/g).mean()))
                rmse.append(float(((p-g)**2).mean().sqrt()))
    tp=conf.diag().double()
    iou=tp/(conf.sum(0)+conf.sum(1)-tp)
    
    return {"mIoU":float(torch.nanmean(iou)),"AbsRel":float(np.mean(absrel)) if absrel else float("nan"),"RMSE":float(np.mean(rmse)) if rmse else float("nan")}


In [9]:
opt=optimizer_for(model); best=float("inf"); history=[]
scaler=torch.amp.GradScaler("cuda",enabled=DEVICE.type=="cuda" and AMP_DTYPE==torch.float16)
total_steps=EPOCHS*len(train_loader); warmup=max(1,int(.05*total_steps))
sched=torch.optim.lr_scheduler.LambdaLR(opt,lambda s:(s+1)/warmup if s<warmup else .01+.99*.5*(1+math.cos(math.pi*(s-warmup)/max(1,total_steps-warmup))))
for epoch in range(1,EPOCHS+1):
    model.train(); running={}; t=time.time()
    for b in tqdm(train_loader,desc=f"epoch {epoch}/{EPOCHS}"):
        b={k:v.to(DEVICE,non_blocking=True) if torch.is_tensor(v) else v for k,v in b.items()}
        opt.zero_grad(set_to_none=True)
        with autocast(): out=model(b["image"]); total,parts=losses(out,b)
        scaler.scale(total).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],1.0); scaler.step(opt); scaler.update(); sched.step()
        for k,v in parts.items(): running[k]=running.get(k,0)+v
    metrics=evaluate(model,val_loader); row={"epoch":epoch,**{f"train_{k}":v/len(train_loader) for k,v in running.items()},**metrics}; history.append(row)
    # Keep the checkpoint with the best validation AbsRel, matching the depth baseline.
    if metrics["AbsRel"]<best:
        best=metrics["AbsRel"]; torch.save({"model":model.state_dict(),"epoch":epoch,"metrics":metrics},RUN_DIR/"shared_multitask_best.pt")
    print(row, "time_s", round(time.time()-t,1))

hist=pd.DataFrame(history); hist.to_csv(RUN_DIR/"shared_multitask_history.csv",index=False); print(hist)

/gorilla/proj/uppmax2026-1-299/taiko_sandbox_please_work/.venv_lusnar/lib/python3.12/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


epoch 1/10:   0%|          | 0/1042 [00:00<?, ?it/s]

{'epoch': 1, 'train_seg': 0.25357345064061615, 'train_depth': 0.38042726143193567, 'mIoU': 0.8604608441134427, 'AbsRel': 0.4844051017132977, 'RMSE': 23.849740928407073} time_s 595.6


epoch 2/10:   0%|          | 0/1042 [00:00<?, ?it/s]

{'epoch': 2, 'train_seg': 0.07012734552148207, 'train_depth': 0.19951022620850928, 'mIoU': 0.8654224358335527, 'AbsRel': 0.46882181497170783, 'RMSE': 21.981593972748954} time_s 549.7


epoch 3/10:   0%|          | 0/1042 [00:00<?, ?it/s]

... (12 outputs left out)

{'epoch': 9, 'train_seg': 0.03185925118074593, 'train_depth': 0.17946043228748434, 'mIoU': 0.889917807903221, 'AbsRel': 0.44488630243621874, 'RMSE': 20.36317665924592} time_s 522.1


epoch 10/10:   0%|          | 0/1042 [00:00<?, ?it/s]

{'epoch': 10, 'train_seg': 0.031130277423005757, 'train_depth': 0.17917216767962743, 'mIoU': 0.889027669406412, 'AbsRel': 0.44098522912829474, 'RMSE': 20.625987106566072} time_s 513.0


   epoch  train_seg  train_depth      mIoU    AbsRel       RMSE
0      1   0.253573     0.380427  0.860461  0.484405  23.849741
1      2   0.070127     0.199510  0.865422  0.468822  21.981594
2      3   0.054260     0.191760  0.872325  0.435826  24.020518
3      4   0.045951     0.187033  0.883611  0.408938  20.320064
4      5   0.041607     0.185266  0.891561  0.462107  21.576018
5      6   0.037790     0.182484  0.889294  0.430292  21.634299
6      7   0.035155     0.180918  0.890323  0.435827  20.718024
7      8   0.032823     0.179839  0.888956  0.445440  20.692497
8      9   0.031859     0.179460  0.889918  0.444886  20.363177
9     10   0.031130     0.179172  0.889028  0.440985  20.625987


In [10]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Model device:", next(model.parameters()).device)

batch = next(iter(train_loader))
print("Batch before transfer:", batch["image"].device)

batch = {
    k: v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v
    for k, v in batch.items()
}
print("Batch after transfer:", batch["image"].device)

model.eval()
with torch.no_grad(), autocast():
    output = model(batch["image"])

print("Segmentation output:", output["seg"].device)
print("Depth output:", output["depth"].device)
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

CUDA available: True
GPU: NVIDIA H100 NVL
Model device: cuda:0


Batch before transfer: cpu
Batch after transfer: cuda:0
Segmentation output: cuda:0
Depth output: cuda:0
GPU memory allocated: 0.95 GB


In [11]:
print("train samples:", len(datasets["train"]))
print("train batches:", len(train_loader))
print(datasets["train"].idx.iloc[0][["rgb", "depth", "label"]])

train samples: 8338
train batches: 1042


rgb      /proj/uppmax2026-1-299/task1_groupc_dataset/lu...
depth    /proj/uppmax2026-1-299/task1_groupc_dataset/lu...
label    /proj/uppmax2026-1-299/task1_groupc_dataset/lu...
Name: 0, dtype: object


In [12]:
%time sample = datasets["train"][0]
print(sample["image"].shape, sample["depth"].shape)


CPU times: user 76.2 ms, sys: 39.1 ms, total: 115 ms
Wall time: 380 ms
torch.Size([3, 512, 512]) torch.Size([1, 512, 512])


In [13]:
%time batch = next(iter(train_loader))
print(batch["image"].shape)

CPU times: user 30.6 ms, sys: 60.7 ms, total: 91.3 ms
Wall time: 3.33 s
torch.Size([8, 3, 512, 512])


In [14]:
b = next(iter(train_loader))
x = b["image"].to(DEVICE)

%time out = model(x)
print(out["seg"].shape, out["depth"].shape)


CPU times: user 14.4 ms, sys: 0 ns, total: 14.4 ms
Wall time: 33.7 ms
torch.Size([8, 5, 512, 512]) torch.Size([8, 1, 512, 512])


In [15]:
ck=torch.load(RUN_DIR/"shared_multitask_best.pt",map_location=DEVICE)
model.load_state_dict(ck["model"]); print("best checkpoint:",ck["epoch"],ck["metrics"])
results={"val":evaluate(model,val_loader)}
if test_loader is not None: results["test"]=evaluate(model,test_loader)
print(json.dumps(results,indent=2))
(RUN_DIR/"shared_multitask_summary.json").write_text(json.dumps({"backbone":DINO_ID,"pretrained":model.pretrained,"out_layers":OUT_LAYERS,"unfreeze_last_n":UNFREEZE_LAST_N,"results":results},indent=2))

best checkpoint: 4 {'mIoU': 0.8836105525021363, 'AbsRel': 0.40893828767392476, 'RMSE': 20.32006424961048}


/gorilla/proj/uppmax2026-1-299/taiko_sandbox_please_work/.venv_lusnar/lib/python3.12/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


{
  "val": {
    "mIoU": 0.8836105525021363,
    "AbsRel": 0.40893828767392476,
    "RMSE": 20.32006424961048
  },
  "test": {
    "mIoU": 0.8358273265465319,
    "AbsRel": 0.3301453581857581,
    "RMSE": 45.237591316802764
  }
}


423